In [ ]:
import yaml
import re
import matplotlib.pyplot as plt
import accelforge as af

# File path
PROB_FILE = 'prob.yaml'

def update_filtering_ratio(yaml_path, new_ratio):
    """Dynamically updates the prob.yaml file to simulate different filtering ratios."""
    with open(yaml_path, 'r') as file:
        data = yaml.safe_load(file)
    
    density = 1.0 / new_ratio
    
    try:
        # Standard Timeloop problem format. 
        # UPDATE 'Inputs' IF YOUR TENSOR IS NAMED SOMETHING ELSE!
        data['problem']['instance']['densities']['Inputs'] = density
    except KeyError:
        print("Warning: Could not find the density key in prob.yaml. Please verify your tensor names.")
    
    with open(yaml_path, 'w') as file:
        yaml.dump(data, file, default_flow_style=False)
    print(f"Updated prob.yaml for a {new_ratio}x filtering ratio (Density: {density:.3f}).")

def evaluate_mapping():
    """Evaluates the mapping directly using AccelForge's native Python API."""
    print("Running AccelForge API...", end=" ", flush=True)
    try:
        # Load the YAMLs directly into the AccelForge specification object
        # AccelForge accepts a list of file paths to build the system
        yaml_files = ["arch.yaml", "components.yaml", "prob.yaml", "map.yaml"]
        spec = af.Spec.from_yaml(yaml_files)
        
        # Evaluate the architecture mapping
        mappings = spec.map_workload_to_arch(print_progress=False, print_number_of_pmappings=False)
        
        # Extract energy directly from the mapping object in memory (returns dict of components)
        energy_dict = mappings.energy(per_component=True)
        
        # Timeloop outputs natively in Joules; scale to pJ (matching your lab's logic)
        energy_pj = sum(energy_dict.values()) * 1e12
        
        # Retrieve cycles (latency)
        try:
            cycles = mappings.cycles()
        except AttributeError:
            try:
                cycles = mappings.latency()
            except AttributeError:
                # Fallback: parse it from the object's string representation if the method alias changes
                stats_str = str(mappings)
                cycles_match = re.search(r'Cycles\s*:\s*([0-9]+)', stats_str)
                cycles = int(cycles_match.group(1)) if cycles_match else 0

        print("Done.")
        return energy_pj, cycles

    except Exception as e:
        print("FAILED!")
        print(f"Error output: {e}")
        return None, None

def main():
    filtering_ratios = [5, 10, 20, 40]
    energies = []
    latencies = []

    for ratio in filtering_ratios:
        # 1. Modify the YAML
        update_filtering_ratio(PROB_FILE, ratio)
        
        # 2 & 3. Run the simulation & Parse the results natively
        energy, cycles = evaluate_mapping()
        
        if energy is None or cycles is None:
            print("Simulation failed. Aborting sweep.")
            break
            
        energies.append(energy)
        latencies.append(cycles)
        
        print(f"Result -> Energy: {energy:.2f} pJ | Latency: {cycles} cycles\n")

    # 4. Visualize the results
    if energies and latencies:
        plt.figure(figsize=(10, 5))
        
        # Plot Energy
        plt.subplot(1, 2, 1)
        plt.plot(filtering_ratios[:len(energies)], energies, marker='o', color='blue')
        plt.title('Total Energy vs. Filtering Ratio')
        plt.xlabel('Filtering Ratio (X times reduction)')
        plt.ylabel('Total Energy (pJ)')
        plt.grid(True)

        # Plot Latency
        plt.subplot(1, 2, 2)
        plt.plot(filtering_ratios[:len(latencies)], latencies, marker='s', color='red')
        plt.title('Latency vs. Filtering Ratio')
        plt.xlabel('Filtering Ratio (X times reduction)')
        plt.ylabel('Latency (Cycles)')
        plt.grid(True)

        plt.tight_layout()
        plt.show()

if __name__ == "__main__":
    main()

Updated prob.yaml for a 5x filtering ratio (Density: 0.200).
Running Timeloop-Model... FAILED!
Error output:

/bin/sh: 1: timeloop-model: not found

Simulation failed. Aborting sweep.


In [8]:
!which timeloop-model
!which accelforge

In [9]:
!find / -name "timeloop-model" -type f 2>/dev/null
!find / -name "accelforge" -type f 2>/dev/null